# Transformer の K・V 計算ステップ デモ

このノートブックでは、Transformer の Self-Attention における **K（Key）・V（Value）・Q（Query）の計算ステップ**を、小さな数値例で可視化します。

$$
\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V
$$

---
**必要ライブラリ**：`numpy`, `matplotlib`（Google Colab にデフォルト搭載）

## セットアップ手順

**初回のみ（セル 1）**: Noto CJK フォントをインストールします。

**カーネル再起動後（毎回）**: セル 1 はスキップして **セル 2（ステップ 0）から実行**してください。  
フォント設定はセル 2 に組み込まれているため、以降のグラフは文字化けしません。


In [ ]:
import subprocess, glob

# Noto CJK JP をインストール（初回のみ時間がかかります）
subprocess.run(['apt-get', 'install', '-y', '-q', 'fonts-noto-cjk'], check=True)

# フォントファイルをキャッシュに登録
import matplotlib.font_manager as fm
for path in glob.glob('/usr/share/fonts/**/*.ttc', recursive=True):
    if 'Noto' in path and 'CJK' in path:
        fm.fontManager.addfont(path)

print('✅ フォントインストール完了 — 次のセルに進んでください')


---
## ステップ 0　ライブラリのインポートとフォント設定

> **カーネル再起動後はここから実行してください。**  
> フォントの登録・適用もこのセルで行います。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import glob

# --- 日本語フォント設定（カーネル再起動後も毎回ここで適用） ---
for path in glob.glob('/usr/share/fonts/**/*.ttc', recursive=True):
    if 'Noto' in path and 'CJK' in path:
        fm.fontManager.addfont(path)

cjk = sorted(set(f.name for f in fm.fontManager.ttflist
               if 'Noto' in f.name and 'CJK' in f.name))
font_name = cjk[0] if cjk else 'DejaVu Sans'
plt.rcParams['font.family']        = font_name
plt.rcParams['axes.unicode_minus'] = False
print(f'フォント: {font_name}')
# -------------------------------------------------------

np.set_printoptions(precision=3, suppress=True)
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False

def show_matrix(ax, M, title, cmap='Blues', vmin=None, vmax=None, fmt='.3f',
                row_labels=None, col_labels=None):
    im = ax.imshow(M, cmap=cmap, aspect='auto',
                   vmin=vmin if vmin is not None else M.min(),
                   vmax=vmax if vmax is not None else M.max())
    ax.set_title(title, fontsize=11, fontweight='bold', pad=8)
    ax.set_xticks(range(M.shape[1]))
    ax.set_yticks(range(M.shape[0]))
    ax.set_xticklabels(col_labels if col_labels else [f'd{i}' for i in range(M.shape[1])], fontsize=9)
    ax.set_yticklabels(row_labels if row_labels else [f'T{i}' for i in range(M.shape[0])], fontsize=9)
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            ax.text(j, i, format(M[i, j], fmt),
                    ha='center', va='center', fontsize=9,
                    color='white' if M[i, j] > (M.max() * 0.6) else 'black')
    return im

def softmax(x, axis=-1):
    e = np.exp(x - x.max(axis=axis, keepdims=True))
    return e / e.sum(axis=axis, keepdims=True)

print('✅ 準備完了')


---
## ステップ 1　入力トークン列 X の定義

- トークン数 `T = 3`（例："The cat sat" = 3 トークン）
- 埋め込み次元 `d_model = 2`（説明のため 2 次元に簡略化）
- 各行が 1 トークンの埋め込みベクトル

In [ ]:
T       = 3   # トークン数
d_model = 2   # 埋め込み次元
d_k     = 2   # Key/Query 次元
d_v     = 2   # Value 次元

X = np.array([
    [1.0, 0.5],  # T0: "The"
    [0.8, 1.2],  # T1: "cat"
    [0.3, 0.9],  # T2: "sat"
])

print(f'X の形状: {X.shape}  (T={T} トークン x d_model={d_model})')
print('\nX =\n', X)

fig, ax = plt.subplots(figsize=(3, 3))
show_matrix(ax, X, '入力 X  (3 x 2)', cmap='Blues',
            row_labels=['T0: The', 'T1: cat', 'T2: sat'],
            col_labels=['d0', 'd1'])
plt.tight_layout()
plt.show()

---
## ステップ 2　重み行列 Wq・Wk・Wv の定義

実際の Transformer では学習によって最適化されます。ここでは固定値を使用します。

In [ ]:
W_q = np.array([[0.2, 0.8], [0.6, 0.4]])
W_k = np.array([[0.4, 0.6], [0.9, 0.1]])
W_v = np.array([[0.3, 0.7], [0.5, 0.2]])

print('W_q =\n', W_q)
print('\nW_k =\n', W_k)
print('\nW_v =\n', W_v)

fig, axes = plt.subplots(1, 3, figsize=(8, 2.5))
show_matrix(axes[0], W_q, 'Wq  (2 x 2)', cmap='Oranges')
show_matrix(axes[1], W_k, 'Wk  (2 x 2)', cmap='Oranges')
show_matrix(axes[2], W_v, 'Wv  (2 x 2)', cmap='Oranges')
plt.suptitle('Weight matrices Wq / Wk / Wv', fontsize=12, fontweight='bold', y=1.04)
plt.tight_layout()
plt.show()

---
## ステップ 3　Q・K・V の計算

$$Q = X W_q, \quad K = X W_k, \quad V = X W_v$$

| 行列 | 意味 |
|------|------|
| Q（Query） | 「何を知りたいか」 |
| K（Key） | 「何を持っているか」 |
| V（Value） | 「何を渡すか」 |

In [ ]:
Q = X @ W_q
K = X @ W_k
V = X @ W_v

print(f'Q =\n{Q}\n\nK =\n{K}\n\nV =\n{V}')

tok = ['The', 'cat', 'sat']
fig, axes = plt.subplots(1, 3, figsize=(9, 3))
show_matrix(axes[0], Q, 'Q = X·Wq  (3 x 2)', cmap='YlOrBr', row_labels=tok)
show_matrix(axes[1], K, 'K = X·Wk  (3 x 2)', cmap='YlOrRd', row_labels=tok)
show_matrix(axes[2], V, 'V = X·Wv  (3 x 2)', cmap='Greens',  row_labels=tok)
plt.suptitle('Q, K, V  projections', fontsize=12, fontweight='bold', y=1.04)
plt.tight_layout()
plt.show()

---
## ステップ 4　アテンションスコアの計算

$$\text{scores} = \frac{Q K^\top}{\sqrt{d_k}}$$

`scores[i][j]` はトークン i がトークン j にどれだけ「注目」するかの生スコアです。  
$\sqrt{d_k}$ で割ることで勾配消失を防ぎます。

In [ ]:
scale  = np.sqrt(d_k)
scores = (Q @ K.T) / scale

print(f'sqrt(d_k) = {scale:.3f}')
print(f'scores =\n{scores}')

tok = ['The', 'cat', 'sat']
fig, axes = plt.subplots(1, 2, figsize=(8, 3))
show_matrix(axes[0], Q @ K.T, 'QK^T（スケーリング前）',        cmap='Reds', row_labels=tok, col_labels=tok)
show_matrix(axes[1], scores,  f'QK^T / √{d_k}（スケーリング後）', cmap='Reds', row_labels=tok, col_labels=tok)
plt.suptitle('Attention scores', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## ステップ 5　Softmax によるアテンション重みの計算

$$\text{attn} = \text{softmax}(\text{scores})$$

各行の合計が 1.0 になり、「注目の割り当て」として解釈できます。

In [ ]:
attn = softmax(scores, axis=-1)

print('attn =\n', attn)
print('各行の合計:', attn.sum(axis=-1))

tok = ['The', 'cat', 'sat']
fig, ax = plt.subplots(figsize=(4.5, 3.5))
im = show_matrix(ax, attn, 'アテンション重み（Softmax 後）',
                 cmap='YlGn', vmin=0, vmax=1, fmt='.3f',
                 row_labels=tok, col_labels=tok)
ax.set_xlabel('Key token (attended to)', fontsize=9)
ax.set_ylabel('Query token (attending)', fontsize=9)
plt.colorbar(im, ax=ax, fraction=0.04)
plt.tight_layout()
plt.show()

---
## ステップ 6　最終出力の計算

$$\text{Output} = \text{attn} \cdot V$$

各トークンの出力は、全トークンの V の加重平均です。

In [ ]:
output = attn @ V

print(f'出力の形状: {output.shape}  <- 入力 X と同じ形状')
print('Output =\n', output)

tok = ['The', 'cat', 'sat']
fig, axes = plt.subplots(1, 3, figsize=(9, 3))
show_matrix(axes[0], attn,   'Attention  (3x3)', cmap='YlGn', vmin=0, vmax=1,
            row_labels=tok, col_labels=tok)
axes[1].axis('off')
axes[1].text(0.5, 0.5, 'x  V  =', ha='center', va='center',
             fontsize=18, fontweight='bold', color='gray', transform=axes[1].transAxes)
show_matrix(axes[2], output, '出力  (3x2)', cmap='Blues',
            row_labels=tok, col_labels=['d0', 'd1'])
plt.suptitle('Attention x V = Output', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## ステップ 7　全ステップの一括まとめ図

In [ ]:
tok = ['The', 'cat', 'sat']

fig, axes = plt.subplots(2, 4, figsize=(14, 6))
fig.suptitle('Transformer Self-Attention: all steps', fontsize=14, fontweight='bold', y=1.01)

show_matrix(axes[0,0], X,      'X  入力 (3x2)',           cmap='Blues',   row_labels=tok)
show_matrix(axes[0,1], Q,      'Q = X·Wq (3x2)',          cmap='YlOrBr',  row_labels=tok)
show_matrix(axes[0,2], K,      'K = X·Wk (3x2)',          cmap='YlOrRd',  row_labels=tok)
show_matrix(axes[0,3], V,      'V = X·Wv (3x2)',          cmap='Greens',  row_labels=tok)
show_matrix(axes[1,0], scores, 'QK^T/√dk スコア (3x3)',   cmap='Reds',    row_labels=tok, col_labels=tok)
show_matrix(axes[1,1], attn,   'Attentionウェイト (3x3)', cmap='YlGn',    vmin=0, vmax=1, row_labels=tok, col_labels=tok)
show_matrix(axes[1,2], output, 'Output = attn·V (3x2)',   cmap='Purples', row_labels=tok, col_labels=['d0','d1'])

axes[1,3].axis('off')
axes[1,3].text(0.05, 0.95,
    "計算フロー\n"
    "──────────\n"
    "(1) X -> Q, K, V\n"
    "    (線形変換)\n\n"
    "(2) QK^T/√dk\n"
    "    (類似度計算)\n\n"
    "(3) softmax\n"
    "    (確率化)\n\n"
    "(4) attn × V\n"
    "    (加重平均)",
    transform=axes[1,3].transAxes, va='top', fontsize=9,
    , color='#333333')

plt.tight_layout()
plt.savefig('transformer_kv_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ transformer_kv_summary.png として保存しました')


---
## ボーナス　関数化してパラメータを自由に変更

`T`（トークン数）・`d_model`（次元）を変えて動作を確認できます。

In [ ]:
def self_attention(X, W_q=None, W_k=None, W_v=None, verbose=True):
    """Scaled Dot-Product Self-Attention の完全実装"""
    T, d = X.shape
    if W_q is None: W_q = np.random.randn(d, d) * 0.5
    if W_k is None: W_k = np.random.randn(d, d) * 0.5
    if W_v is None: W_v = np.random.randn(d, d) * 0.5
    Q_ = X @ W_q
    K_ = X @ W_k
    V_ = X @ W_v
    a  = softmax((Q_ @ K_.T) / np.sqrt(d), axis=-1)
    if verbose:
        print(f'入力: {X.shape}  Q/K/V: {Q_.shape}  Attention: {a.shape}')
    return a @ V_, a

# ---- ここを変更して試してみましょう ----
T_new = 5   # トークン数
d_new = 4   # 埋め込み次元
np.random.seed(42)
# ----------------------------------------

X_new = np.random.randn(T_new, d_new)
_, attn_new = self_attention(X_new)

fig, ax = plt.subplots(figsize=(4.5, 3.5))
im = show_matrix(ax, attn_new,
                 f'アテンション重み  (T={T_new}, d={d_new})',
                 cmap='YlGn', vmin=0, vmax=attn_new.max(), fmt='.2f',
                 row_labels=[f'T{i}' for i in range(T_new)],
                 col_labels=[f'T{i}' for i in range(T_new)])
plt.colorbar(im, ax=ax, fraction=0.04)
plt.tight_layout()
plt.show()

---
## まとめ

| ステップ | 演算 | 意味 |
|---------|------|------|
| 1 | `Q = X @ Wq` | 「何を知りたいか」のベクトル生成 |
| 2 | `K = X @ Wk` | 「何を持っているか」のベクトル生成 |
| 3 | `V = X @ Wv` | 「何を渡すか」の情報ベクトル生成 |
| 4 | `scores = QK^T / √dk` | Q と K の類似度スコア計算 |
| 5 | `attn = softmax(scores)` | スコアを確率（注目度）に変換 |
| 6 | `output = attn @ V` | 注目度で重み付けした情報の合成 |

> **ポイント**：LLM の「文脈理解」は、本質的にこの加重平均の繰り返しです。  
> どのトークンにどれだけ注目するかを、Q・K の類似度が決定しています。